# Day 066 | Loss Functions & Optimizers
## 100 Days of ML for Geotechnical Engineering

| | |
|---|---|
| **Author** | Ripon Chandra Malo |
| **Phase** | 4 — Deep Learning |
| **Topic** | Loss Functions & Optimizers |
| **Tools** | PyTorch, NumPy, Matplotlib |
| **Geo Problem** | Choosing the right loss and optimizer for settlement prediction tasks |

---

## Concept Overview

| Concept | Description | Geotechnical Use Case |
|---|---|---|
| MSELoss | Mean squared error; penalizes large errors heavily | Settlement regression (clean data) |
| L1Loss (MAE) | Mean absolute error; robust to outliers | Settlement regression (outlier-prone field data) |
| HuberLoss | Blend of MSE and MAE; quadratic near zero, linear in tails | General regression with occasional bad readings |
| BCEWithLogitsLoss | Binary cross-entropy with sigmoid; numerically stable | Liquefaction yes/no classification |
| CrossEntropyLoss | Multi-class classification loss | Soil type prediction (sand/clay/silt/gravel) |
| SGD | Stochastic gradient descent; stable but slow | Baseline optimizer |
| Adam | Adaptive moment estimation; fast convergence | Most geo-ML tasks |
| AdamW | Adam with proper weight decay decoupling | Regularized settlement models |
| StepLR | Multiply LR by gamma every N epochs | Scheduled decay training |
| ReduceLROnPlateau | Reduce LR when val-loss stagnates | Early adaptive learning |

---

## Geotechnical Problem Framing

In geotechnical engineering we frequently fit neural networks to predict:
- **Settlement (mm)** — a continuous regression target, sometimes with outliers from poor sensor readings
- **Liquefaction potential** — a binary yes/no classification task
- **Soil type** — a multi-class classification task (sand, clay, silt, gravel)

Choosing the wrong loss function leads to either ignoring field outliers or over-fitting to noise.
Choosing the wrong optimizer leads to slow or unstable training on limited geotechnical datasets.

Today we build intuition for all of these choices with synthetic settlement data.

---
## Part 1 | Setup & Synthetic Settlement Data

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 1 — Setup & Synthetic Settlement Data
# ─────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

# Synthetic settlement dataset
# Features: [applied_load_kPa, soil_compressibility, layer_thickness_m]
# Target:   settlement_mm
N = 100
load        = np.random.uniform(50, 300, N)        # kPa
compress    = np.random.uniform(0.01, 0.05, N)     # Cc
thickness   = np.random.uniform(2, 10, N)          # m

# Physics-inspired relationship + noise
settlement  = 0.5 * load * compress * thickness + np.random.normal(0, 5, N)

# Inject a few outliers (bad field readings)
outlier_idx = np.random.choice(N, 5, replace=False)
settlement[outlier_idx] += np.random.uniform(60, 100, 5)

# Convert to tensors
X = torch.tensor(np.column_stack([load, compress, thickness]), dtype=torch.float32)
y = torch.tensor(settlement, dtype=torch.float32).view(-1, 1)

# Normalize features
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / X_std

print(f"Dataset shape: X={X_norm.shape}, y={y.shape}")
print(f"Settlement range: {settlement.min():.1f} — {settlement.max():.1f} mm")
print(f"Outlier indices: {outlier_idx}")

---
## Part 2 | Comparing Loss Functions: MSE vs L1 vs Huber

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 2 — Loss Function Comparison (Visualisation)
# ─────────────────────────────────────────────────────────
errors = torch.linspace(-60, 60, 300)

mse_loss   = errors ** 2
l1_loss    = errors.abs()
huber_loss = torch.where(errors.abs() < 1,
                         0.5 * errors**2,
                         errors.abs() - 0.5)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(errors.numpy(), mse_loss.numpy(),   label='MSELoss (L2)',   lw=2)
ax.plot(errors.numpy(), l1_loss.numpy(),    label='L1Loss (MAE)',   lw=2, linestyle='--')
ax.plot(errors.numpy(), huber_loss.numpy(), label='HuberLoss',      lw=2, linestyle=':')
ax.set_xlabel('Prediction Error (mm)')
ax.set_ylabel('Loss Value')
ax.set_title('Loss Functions — Sensitivity to Outliers\n(Geotechnical: Settlement Prediction Error)')
ax.legend()
ax.set_ylim(0, 200)
ax.grid(True, alpha=0.3)

# Annotate the key insight
ax.annotate('MSE explodes\nfor large errors', xy=(45, 45**2), xytext=(20, 1800),
            fontsize=8, color='C0',
            arrowprops=dict(arrowstyle='->', color='C0'))

plt.tight_layout()
plt.show()

print("KEY INSIGHT:")
print("  MSELoss   — punishes large errors (outliers) quadratically => bad for noisy field data")
print("  L1Loss    — treats all errors linearly => robust but gradient is constant near zero")
print("  HuberLoss — MSE near zero, L1 in tails => best of both worlds for geo data")

---
## Part 3 | Classification Losses: BCE and CrossEntropy

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 3 — Classification Losses
# ─────────────────────────────────────────────────────────

# --- Binary: Liquefaction yes/no ---
bce_loss_fn = nn.BCEWithLogitsLoss()

# Simulate raw logits for 6 samples
logits_bin  = torch.tensor([ 2.0,  1.5, -1.0, -2.0,  0.5, -0.5])  # positive => liquefaction
labels_bin  = torch.tensor([ 1.0,  1.0,  0.0,  0.0,  1.0,  0.0])  # ground truth

bce_val = bce_loss_fn(logits_bin, labels_bin)
print(f"BCEWithLogitsLoss for liquefaction classification: {bce_val.item():.4f}")
probs = torch.sigmoid(logits_bin)
print(f"  Predicted probabilities: {probs.detach().numpy().round(2)}")
print(f"  True labels:             {labels_bin.numpy().astype(int)}")

print()

# --- Multi-class: Soil type (sand=0, clay=1, silt=2, gravel=3) ---
ce_loss_fn  = nn.CrossEntropyLoss()

# 4 samples, 4 classes — raw logits
logits_mc   = torch.tensor([
    [ 3.0,  0.5, -0.5, -1.0],   # likely sand
    [-1.0,  2.5,  0.0,  0.2],   # likely clay
    [ 0.1, -0.3,  2.8,  0.0],   # likely silt
    [-0.5,  0.0, -0.5,  3.1],   # likely gravel
])
labels_mc   = torch.tensor([0, 1, 2, 3])   # ground truth soil types

ce_val = ce_loss_fn(logits_mc, labels_mc)
print(f"CrossEntropyLoss for soil type classification: {ce_val.item():.4f}")
probs_mc = torch.softmax(logits_mc, dim=1)
print(f"  Softmax probabilities (sand | clay | silt | gravel):")
soil_labels = ['sand', 'clay', 'silt', 'gravel']
for i, (row, gt) in enumerate(zip(probs_mc, labels_mc)):
    pred = soil_labels[row.argmax().item()]
    gt_s = soil_labels[gt.item()]
    print(f"    Sample {i}: pred={pred:6s} | true={gt_s:6s} | {row.detach().numpy().round(2)}")

---
## Part 4 | Building a Settlement Regression Network

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 4 — Settlement Regression Network
# ─────────────────────────────────────────────────────────

class SettlementNet(nn.Module):
    """Simple fully-connected network for settlement regression.
    Input: [load, compressibility, thickness]  (normalised)
    Output: settlement_mm
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

# Quick architecture summary
model_test = SettlementNet()
total_params = sum(p.numel() for p in model_test.parameters())
print(model_test)
print(f"\nTotal parameters: {total_params}")

---
## Part 5 | Optimizer Comparison: SGD vs Adam vs AdamW

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 5 — SGD vs Adam vs AdamW Convergence
# ─────────────────────────────────────────────────────────

dataset    = TensorDataset(X_norm, y)
loader     = DataLoader(dataset, batch_size=32, shuffle=True)
loss_fn    = nn.HuberLoss(delta=10.0)   # robust to outlier settlements
EPOCHS     = 150

def train_model(optimizer_name):
    torch.manual_seed(0)
    model = SettlementNet()
    lr    = 1e-2

    if optimizer_name == 'SGD':
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_name == 'Adam':
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    else:   # AdamW
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    history = []
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(xb)
        history.append(epoch_loss / N)
    return history

print("Training with SGD, Adam, AdamW ...")
histories = {name: train_model(name) for name in ['SGD', 'Adam', 'AdamW']}

fig, ax = plt.subplots(figsize=(9, 4))
for name, hist in histories.items():
    ax.plot(hist, label=name, lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('HuberLoss')
ax.set_title('Optimizer Convergence — Settlement Regression\n(SGD vs Adam vs AdamW)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

for name, hist in histories.items():
    print(f"  {name:5s} — final loss: {hist[-1]:.3f}")

---
## Part 6 | Learning Rate Schedulers

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 6 — LR Schedulers: StepLR and ReduceLROnPlateau
# ─────────────────────────────────────────────────────────

def train_with_scheduler(scheduler_type):
    torch.manual_seed(0)
    model = SettlementNet()
    opt   = torch.optim.Adam(model.parameters(), lr=1e-2)

    if scheduler_type == 'StepLR':
        scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.5)
    else:   # ReduceLROnPlateau
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10,
                                                                factor=0.5, verbose=False)

    history  = []
    lr_trace = []
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(xb)
        avg_loss = epoch_loss / N
        history.append(avg_loss)
        lr_trace.append(opt.param_groups[0]['lr'])

        if scheduler_type == 'StepLR':
            scheduler.step()
        else:
            scheduler.step(avg_loss)
    return history, lr_trace

hist_step, lr_step   = train_with_scheduler('StepLR')
hist_plat, lr_plat   = train_with_scheduler('ReduceLROnPlateau')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, hist, lr, name in zip(axes,
                               [hist_step, hist_plat],
                               [lr_step, lr_plat],
                               ['StepLR', 'ReduceLROnPlateau']):
    ax2 = ax.twinx()
    ax.plot(hist, color='C0', label='Loss', lw=2)
    ax2.plot(lr, color='C3', linestyle='--', label='LR', lw=1.5, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('HuberLoss', color='C0')
    ax2.set_ylabel('Learning Rate', color='C3')
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

fig.suptitle('LR Schedulers — Settlement Regression Training', y=1.02)
plt.tight_layout()
plt.show()

print("StepLR:             reduces LR by 50% every 30 epochs — predictable schedule")
print("ReduceLROnPlateau:  reduces LR when training stalls — adaptive to data")

---
## Part 7 | Comparing Loss Functions on Settlement Regression

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 7 — MSE vs L1 vs Huber on Settlement Data
# ─────────────────────────────────────────────────────────

def train_loss_fn(loss_name):
    torch.manual_seed(0)
    model = SettlementNet()
    opt   = torch.optim.Adam(model.parameters(), lr=1e-2)

    fns = {
        'MSE':   nn.MSELoss(),
        'L1':    nn.L1Loss(),
        'Huber': nn.HuberLoss(delta=10.0)
    }
    fn = fns[loss_name]

    history = []
    for _ in range(EPOCHS):
        epoch_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = fn(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(xb)
        history.append(epoch_loss / N)

    # Evaluate final MAE in mm
    model.eval()
    with torch.no_grad():
        preds = model(X_norm)
        mae   = (preds - y).abs().mean().item()
    return history, mae

results = {}
for name in ['MSE', 'L1', 'Huber']:
    hist, mae = train_loss_fn(name)
    results[name] = (hist, mae)

print("Loss Function Comparison (trained with outlier-injected data):")
print(f"{'Loss Fn':8s} | Final MAE (mm)")
print("-" * 25)
for name, (hist, mae) in results.items():
    print(f"{name:8s} | {mae:.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
for name, (hist, _) in results.items():
    ax.plot(hist, label=name, lw=2)
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss (log scale)')
ax.set_title('Loss Function Training Curves — Settlement (with outliers)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Practice Exercises

1. **Loss exploration** — Change the HuberLoss `delta` parameter to 1.0 and 50.0. How does this affect training stability and final MAE?
2. **Optimizer tuning** — Try SGD without momentum. Does it converge within 150 epochs?
3. **Scheduler experiment** — Use `CosineAnnealingLR` instead of `StepLR`. Plot the LR trace and compare final loss.
4. **Classification task** — Build a BCEWithLogitsLoss model to predict liquefaction (binary) from synthetic SPT-N, unit weight, and water table depth features.
5. **Learning rate sensitivity** — Train with Adam at lr=0.001, 0.01, 0.1. Which converges fastest without diverging?

In [ ]:
# ─────────────────────────────────────────────────────────
# PRACTICE — Starter code (complete the exercises above)
# ─────────────────────────────────────────────────────────

# Exercise 1: Change HuberLoss delta
# for delta in [1.0, 10.0, 50.0]:
#     loss_fn = nn.HuberLoss(delta=delta)
#     hist, mae = train_loss_fn_custom(loss_fn)
#     print(f"delta={delta:5.1f} -> MAE={mae:.2f} mm")

# Exercise 4: Liquefaction binary classifier
# np.random.seed(0)
# spt_n     = np.random.uniform(5, 40, 200)
# gamma     = np.random.uniform(16, 21, 200)   # kN/m3
# water_tbl = np.random.uniform(0.5, 5.0, 200) # m depth
# liq_label = (spt_n < 20).astype(float)       # simplified rule
# ...

print("Practice exercise starters are ready — uncomment and complete!")

---
## ML Connection Table

| Geotechnical Task | PyTorch Loss | Optimizer Choice | Scheduler |
|---|---|---|---|
| Settlement regression (clean data) | MSELoss | Adam | StepLR |
| Settlement regression (noisy field) | HuberLoss | AdamW | ReduceLROnPlateau |
| Liquefaction risk (binary) | BCEWithLogitsLoss | Adam | None or StepLR |
| Soil classification (multi-class) | CrossEntropyLoss | Adam | CosineAnnealingLR |
| Bearing capacity with physical priors | L1Loss | SGD+momentum | StepLR |

---
## Key Takeaways

1. **MSELoss** amplifies outliers quadratically — avoid it when field measurements have sensor errors. Use **HuberLoss** as the default for geotechnical regression.
2. **BCEWithLogitsLoss** numerically combines sigmoid + BCE in one step and is the correct choice for binary geotechnical classification (liquefaction, failure/no-failure).
3. **Adam and AdamW** converge much faster than plain SGD on small geo datasets; AdamW's proper weight decay prevents overfitting on limited samples.
4. **ReduceLROnPlateau** is especially useful in geotechnical applications where training loss plateaus due to noisy data — it adapts the learning rate automatically.
5. Scheduler and loss function choices are as important as network architecture for geotechnical deep learning.

---
*Day 066 of 100 — Every loss function is a hypothesis about your data. Choose wisely.*